# Offline investigation
Build, verify, inspect and export a five-event synthetic case. No API key or network call is used. Install the project with the notebooks extra first.

In [ ]:
from pathlib import Path
import tempfile
import json
root = Path.cwd()
if not (root / 'examples').exists():
    root = root.parent
assert (root / 'examples/parser_samples.json').exists(), 'Run from the repository or notebooks directory'
from timeline_demo.pipeline import Input, run_pipeline, read_timeline
from timeline_demo.core.manifest import verify_bundle
work = tempfile.TemporaryDirectory()
workdir = Path(work.name)
bundle = workdir / 'bundle'
manifest = run_pipeline([
    Input('cloudtrail', root / 'examples/raw/aws/cloudtrail_real_sample.json'),
    Input('entra_signin', root / 'examples/raw/entra/entra_signin_real_sample.jsonl'),
    Input('crowdstrike_detection', root / 'examples/raw/edr/crowdstrike_detection_real_sample.json'),
], bundle, 'notebook-demo')


In [ ]:
checked = verify_bundle(bundle)
assert checked['counts']['event_count'] == 5
checked['counts']

In [ ]:
events = list(read_timeline(bundle))
[(e['time_utc'], e['source_name'], e['activity_name']) for e in events]

## Inspect a source reference
The record hash covers deterministic JSON encoding. The file hash covers the archived original bytes. Neither hash proves that a source told the truth.

In [ ]:
event = events[0]
{'event':event, 'raw_source':str(bundle / event['evidence_path'])}

In [ ]:
json.loads((bundle/'extracted_iocs.json').read_text())

## Demonstrate tamper detection on a disposable copy

In [ ]:
import shutil
altered = workdir/'altered'
shutil.copytree(bundle, altered)
(altered/'timeline.jsonl').write_text('{}\n')
try:
    verify_bundle(altered)
    raise AssertionError('tampering was not detected')
except ValueError as error:
    print(type(error).__name__, str(error))

In [ ]:
work.cleanup()